# 🔬 SpectraMatch — Interactive Chemometrics Walkthrough

Welcome to **SpectraMatch**, an algorithmic platform for the chemometric analysis, quality control, and identification of complex **Natural Organic Matter (NOM/DOM)** and environmental isolates.

This notebook demonstrates the end-to-end analytical pipeline:
1. **Synthetic Data Generation & Loading**
2. **Asymmetric Least Squares (AsLS) Baseline Subtraction**
3. **6-Stage Analytical Quality Control (QC Suite)**
4. **Chemometric Functional Indices Extraction**
5. **In-Memory Vector Search against IHSS Reference Standards**
6. **Chemical Space Mapping via Principal Component Analysis (PCA)**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import core SpectraMatch modules
from core.models import ModalityEnum, MetricEnum
from core.synthetic import generate_synthetic_ftir
from core.normalize import normalize_ftir, _asls_baseline, CANONICAL_FTIR_GRID
from core.qc import run_qc
from core.indices import compute_ftir_indices
from core.similarity import search
from core.chemspace import fit_library_pca, project_query_to_pca

print("✅ SpectraMatch modules successfully loaded!")

## 1. Raw Spectrum & AsLS Baseline Correction

We generate a synthetic FTIR absorbance spectrum representing an unknown aquatic fulvic acid with a baseline drift.

In [ ]:
wn_raw, abs_raw = generate_synthetic_ftir("fulvic_acid", seed=777, noise_sigma=0.01)

# Compute AsLS baseline: lam=1e6, p=0.01
baseline = _asls_baseline(abs_raw, lam=1e6, p=0.01)
corrected = abs_raw - baseline

plt.figure(figsize=(10, 4))
plt.plot(wn_raw, abs_raw, label="Raw FTIR Signal", color="#64748B")
plt.plot(wn_raw, baseline, label="Fitted AsLS Baseline", color="#EF4444", linestyle="--")
plt.plot(wn_raw, corrected, label="Corrected Signal", color="#0284C7")
plt.gca().invert_xaxis()  # Spectroscopic standard (4000 to 400)
plt.xlabel("Wavenumber (cm⁻¹)")
plt.ylabel("Absorbance")
plt.title("FTIR Preprocessing: Asymmetric Least Squares Baseline Correction")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Quality Control & Functional Index Extraction

In [ ]:
# Run QC Suite
qc = run_qc(wn_raw, abs_raw, ModalityEnum.FTIR)
print(f"QC Status: {'PASSED' if qc.passed else 'FAILED'} | Quality Score: {qc.score:.2f}")
print(f"Estimated SNR: {qc.snr_estimate} | Canonical Coverage: {qc.coverage_pct}%")

# Compute Chemometric Indices
indices = compute_ftir_indices(wn_raw, abs_raw)
for k, v in indices.items():
    print(f"  • {k:<15}: {v}")

## 3. High-Throughput In-Memory Library Search

We normalize the spectrum onto the canonical grid (1801 points) and search against precomputed library vectors.

In [ ]:
canonical_query = normalize_ftir(wn_raw, abs_raw)

# Load library vectors and metadata
vectors_data = np.load("library/vectors.npz", allow_pickle=True)
ftir_matrix = vectors_data["ftir_matrix"]
ftir_ids = list(vectors_data["ftir_ids"])

meta_df = pd.read_json("library/metadata.json")
meta_dict = {row["id"]: row for _, row in meta_df.iterrows()}
library_meta = [meta_dict.get(sid, {"name": sid, "sample_type": "unknown"}) for sid in ftir_ids]

# Run search with Pearson correlation
results = search(
    query_vec=canonical_query,
    library_matrix=ftir_matrix,
    library_meta=library_meta,
    modality=ModalityEnum.FTIR,
    metric=MetricEnum.PEARSON,
    top_k=5,
)

print("\n--- Top Library Matches ---")
for r in results:
    print(f"Rank #{r.rank}: {r.name:<40} | Score ({r.metric}) = {r.score:.4f} | Type = {r.sample_type}")

## 4. PCA Chemical Space Projection

In [ ]:
pca = fit_library_pca(ftir_matrix, modality=ModalityEnum.FTIR, n_components=2)
query_pca = project_query_to_pca(canonical_query, pca)

coords = pca.library_coords
plt.figure(figsize=(9, 6))
plt.scatter(coords[:, 0], coords[:, 1], c="#94A3B8", alpha=0.7, edgecolors="none", s=60, label="Library Standards")
plt.scatter(query_pca[0], query_pca[1], c="#0284C7", marker="*", s=220, edgecolors="#0369A1", label="⭐ Query Sample")

plt.xlabel(f"PC1 ({pca.explained_variance_ratio[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio[1]*100:.1f}% var)")
plt.title("PCA Chemical Space Projection")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()